# TODO where's the description?

In [ ]:
# Imports necessary to execute the code
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile
import torch
from careamics.config import GaussianMixtureNMConfig, create_pn2v_configuration
from careamics.lightning import (
    create_predict_datamodule,
    create_train_datamodule,
)
from careamics.lightning.lightning_module import FCNModule
from careamics.models.lvae.noise_models import (
    GaussianMixtureNoiseModel,
    create_histogram,
)
from careamics.prediction_utils import convert_outputs_pn2v
from careamics.utils.metrics import scale_invariant_psnr
from careamics_portfolio import PortfolioManager
from PIL import Image
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint

## Import the dataset

The dataset can be directly downloaded using the `careamics-portfolio` package, which
uses `pooch` to download the data.

In [ ]:
# instantiate data portfolio manage
portfolio = PortfolioManager()

# and download the data
root_path = Path("./data")
files = portfolio.denoising.Convallaria.download(root_path)

# create paths for the data
data_path = Path(root_path / "denoising-Convallaria.unzip/Convallaria_diaphragm")

## Load and visualize data for Noise Model Training

In [ ]:
observation = tifffile.imread(data_path / '20190726_tl_50um_500msec_wf_130EM_FD.tif')

# The data contains 100 images of a static sample.
# We estimate the clean signal by averaging all images.
signal = np.mean(observation[:, ...],axis=0)[np.newaxis, ...]

# Let's look the raw data and our pseudo ground truth signal
plt.figure(figsize=(12, 12))
plt.subplot(1, 2, 2)
plt.title(label='average (ground truth)')
plt.imshow(signal[0],cmap='gray')
plt.subplot(1, 2, 1)
plt.title(label='single raw image')
plt.imshow(observation[0],cmap='gray')
plt.show()

## Creating the GMM noise model


In [ ]:
# train Noise Model for current channel

noise_model_config = GaussianMixtureNMConfig(
    model_type="GaussianMixtureNoiseModel",
    min_signal=signal.min(),
    max_signal=signal.max(),
    n_coeff=3,
    n_gaussian=3,
)
noise_model = GaussianMixtureNoiseModel(noise_model_config)
noise_model.fit(signal=signal, observation=observation, n_epochs=1000)

# save result on disk for later re-use
noise_model.save(path="noise_models/", name="noise_model")

# show the result
histogram = create_histogram(
    bins=100,
    min_val=signal.min(),
    max_val=signal.max(),
    signal=signal,
    observation=observation,
)

In [ ]:
signal_bin_index = 50
channel = 0
n_bins = 100

bin_size = (noise_model.max_signal.item() - noise_model.min_signal.item()) / n_bins
query_signal = (signal_bin_index / n_bins) * (noise_model.max_signal - noise_model.min_signal) + noise_model.min_signal + bin_size / 2

query_observations = torch.arange(noise_model.min_signal.item(), noise_model.max_signal.item(), bin_size) + bin_size / 2

noise_model.mode = "inference"
noise_model.min_signal = torch.tensor(noise_model.min_signal, device=noise_model.device)
noise_model.max_signal = torch.tensor(noise_model.max_signal, device=noise_model.device)

probabilities = noise_model.likelihood(observations=query_observations, signals=torch.tensor(query_signal)).cpu().detach().numpy()

plt.figure(figsize=(12, 5))
plt.suptitle(f"Noise model for channel {channel}")

plt.subplot(1, 2, 1)
plt.xlabel("Observation Bin")
plt.ylabel("Signal Bin")
plt.imshow(histogram[0]**0.25, cmap="gray")
plt.axhline(y=signal_bin_index + 0.5, linewidth=5, color="blue", alpha=0.5)

plt.subplot(1, 2, 2)
plt.plot(query_observations, probabilities, label=f"GMM signal = {query_signal:.2f}", marker=".", color="red", linewidth=2)
plt.xlabel(f"Observations (x) for signal s = {query_signal:.2f}")
plt.ylabel("Probability Density")
plt.title(f"Probability Distribution P(x|s) at signal = {query_signal:.2f}")
plt.legend()

plt.show()

## Load and visualize the traning data


In [ ]:
data = tifffile.imread(data_path / '20190520_tl_25um_50msec_05pc_488_130EM_Conv.tif')

In [ ]:
slice_idx = 0
slice_img = data[slice_idx]
h, w = slice_img.shape

crop_size = 128
num_crops = 3

# Ensure we don't go out of bounds
max_y = h - crop_size
max_x = w - crop_size

crop_coords = [
    (
        np.random.randint(0, max_y + 1),
        np.random.randint(0, max_x + 1)
    )
    for _ in range(num_crops)
]

fig, axes = plt.subplots(1, num_crops + 1, figsize=(20, 5))

# Show the full slice
axes[0].imshow(slice_img, cmap='gray')
axes[0].set_title(f"Slice {slice_idx} (full)")
axes[0].axis('off')

# Show the crops
for i, (y, x) in enumerate(crop_coords):
    crop = slice_img[y:y+crop_size, x:x+crop_size]
    axes[i+1].imshow(crop, cmap='gray')
    axes[i+1].set_title(f"Crop {i+1}: y={y}, x={x}")
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()


## Train with the CAREamics Lightning API

Using the Lightning API of CAREamics, you need to instantiate the lightning module, the 
data module and the trainer yourself.

### Create the Lightning module

In [ ]:
config = create_pn2v_configuration(
    experiment_name="Convallaria_pn2v",
    data_type="array",
    axes="SYX",
    patch_size=(64, 64),
    batch_size=64,
    num_epochs=50,
    nm_path="./noise_models/noise_model.npz",
    train_dataloader_params={"shuffle": True, "num_workers": 4},
    val_dataloader_params={"shuffle": False, "num_workers": 4},
)

### Create the model

In [ ]:
model = FCNModule(config.algorithm_config)

### Create the data module

In [ ]:
train_data_module = create_train_datamodule(
    train_data=data,
    data_type=config.data_config.data_type,
    patch_size=config.data_config.patch_size,
    axes=config.data_config.axes,
    batch_size=config.data_config.batch_size,
    train_dataloader_params=config.data_config.train_dataloader_params,
    val_dataloader_params=config.data_config.val_dataloader_params,
)

### Create the trainer

Note that here we modify the prediction loop, but this will be  changed in the near
future.

In [ ]:
# Create Callbacks
root = Path("Convallaria_pn2v")
callbacks = [
    ModelCheckpoint(
        dirpath=root / "checkpoints",
        filename="convallaria_pn2v_lightning_api",
        save_last=True,
    )
]

# Create a Lightning Trainer
trainer = Trainer(
    max_epochs=config.training_config.lightning_trainer_config["max_epochs"],
    default_root_dir=root,
    callbacks=callbacks,
)

### Train the model

In [ ]:
trainer.fit(model, datamodule=train_data_module)

### Load from checkpoint (Optional)

In [ ]:
ckpt_path = root / "checkpoints" / "convallaria_pn2v_lightning_api.ckpt"
ckpt_dict = torch.load(ckpt_path, map_location="cuda", weights_only=True)
model.load_state_dict(ckpt_dict['state_dict'], strict=True)

## Predict with CAREamics Lightning API

### Define the prediction datamodule

In [ ]:
# Calculate data statistics if training was not performed
if not hasattr(train_data_module, "train_dataset"):
    train_data_module.setup()

means, stds = train_data_module.get_data_statistics()
pred_data_module = create_predict_datamodule(
    pred_data=data[::10, :512, :512],
    data_type="array",
    axes="SYX",
    batch_size=16,
    tta_transforms=True,
    image_means=means,
    image_stds=stds,
    tile_size=(128, 128),
    tile_overlap=(32, 32),
)

### Predict

In [ ]:
predictions = trainer.predict(model, datamodule=pred_data_module)

In [ ]:
# Convert the outputs to the original format, mostly useful if tiling is used
predictions_avg, mse_estimates = convert_outputs_pn2v(predictions, tiled=True)

### Visualize the prediction

In [ ]:
crop_size = 128
slice_idx = 0

input_data = data[slice_idx, :512, :512]
pred_img = predictions_avg[slice_idx].squeeze()
mse_img = mse_estimates[slice_idx].squeeze()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(input_data[:crop_size, :crop_size], cmap="gray")
axes[0, 0].set_title("Input (crop)")
axes[0, 0].axis("off")

axes[0, 1].imshow(pred_img[:crop_size, :crop_size], cmap="gray")
axes[0, 1].set_title("Prediction (crop)")
axes[0, 1].axis("off")

axes[0, 2].imshow(mse_img[:crop_size, :crop_size], cmap="gray")
axes[0, 2].set_title("MMSE Estimate (crop)")
axes[0, 2].axis("off")

axes[1, 0].imshow(input_data, cmap="gray")
axes[1, 0].set_title("Input (full)")
axes[1, 0].axis("off")

axes[1, 1].imshow(pred_img, cmap="gray")
axes[1, 1].set_title("Prediction (full)")
axes[1, 1].axis("off")

axes[1, 2].imshow(mse_img, cmap="gray")
axes[1, 2].set_title("MMSE Estimate (full)")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

### Compute metrics

In [ ]:
psnrs_prior = np.zeros((len(predictions_avg), 1))
psnrs_mse = np.zeros((len(mse_estimates), 1))
gt = np.mean(data[::10, ...], axis=0, keepdims=True)[: , :512, :512]

for i, (prior, mse) in enumerate(zip(predictions_avg, mse_estimates)):
    psnrs_prior [i] = scale_invariant_psnr(gt.squeeze(), prior.squeeze())
    psnrs_mse[i] = scale_invariant_psnr(gt.squeeze(), mse.squeeze())

print(f"PSNR prior: {psnrs_prior.mean():.2f} +/- {psnrs_prior.std():.2f}")
print(f"PSNR mse: {psnrs_mse.mean():.2f} +/- {psnrs_mse.std():.2f}")